# Titanic survival model

This notebook follows one reproducible path from data inspection to model evaluation and final submission.

The experiments kept here are the useful ones: Title, Sex × Pclass, and FarePerPerson. The completed Deck experiment remains archived in `main_deck_experiment.ipynb`.

## 1. Load and inspect the data

In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
train.info()

print("\nMissing values in train:")
print(train.isnull().sum())

print("\nMissing values in test:")
print(test.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB

Missing values in train:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtyp

## 2. Feature engineering and the train/validation split

These transformations are row-wise. The model-dependent operations—imputation, scaling, and one-hot encoding—are kept inside the pipeline.

In [4]:
def make_features(df):
    df = df.copy()

    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]+)\.")[0]
        .str.strip()
    )

    common_titles = ["Mr", "Miss", "Mrs", "Master"]
    df["Title"] = df["Title"].where(
        df["Title"].isin(common_titles),
        "Rare"
    )

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    df["SexPclass"] = df["Sex"] + "_Pclass_" + df["Pclass"].astype(str)
    df["HasCabin"] = df["Cabin"].notna().astype(int)

    return df.drop(
        columns=["PassengerId", "Name", "Ticket", "Cabin"]
    )

In [5]:
X = make_features(train.drop(columns="Survived"))
y = train["Survived"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

Training rows: 712
Validation rows: 179


## 3. Inspect the Title feature

In [6]:
title_summary = (
    X_train
    .assign(Survived=y_train)
    .groupby("Title")["Survived"]
    .agg(count="size", survival_rate="mean")
    .sort_values("count", ascending=False)
)

title_summary["survival_rate"] = title_summary["survival_rate"].round(3)
title_summary

,count,survival_rate
Title,,
Mr,412,0.153
Miss,141,0.702
Mrs,107,0.785
Master,31,0.548
Rare,21,0.476


In [7]:
pd.crosstab(X_train["Title"], X_train["Sex"])

Sex,female,male
Title,,
Master,0,31
Miss,141,0
Mr,0,412
Mrs,107,0
Rare,5,16


In [8]:
raw_title_summary = (
    train.loc[X_train.index, ["Name", "Survived"]]
    .assign(
        RawTitle=lambda df: (
            df["Name"]
            .str.extract(r",\s*([^.]+)\.")[0]
            .str.strip()
        )
    )
    .groupby("RawTitle")["Survived"]
    .agg(count="size", survival_rate="mean")
    .sort_values("count", ascending=False)
)

raw_title_summary["survival_rate"] = raw_title_summary["survival_rate"].round(3)
raw_title_summary

,count,survival_rate
RawTitle,,
Mr,412,0.153
Miss,141,0.702
Mrs,107,0.785
Master,31,0.548
Dr,6,0.500
Rev,5,0.000
Col,2,0.500
Mlle,2,1.000
Don,1,0.000


## 4. Define preprocessing and models

In [9]:
numeric_features = [
    "Pclass", "Age", "SibSp", "Parch",
    "Fare", "FarePerPerson", "FamilySize", "IsAlone", "HasCabin"
]

categorical_features = [
    "Sex", "Embarked", "Title", "SexPclass"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=3, random_state=42))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(max_depth=3, random_state=42))
])

## 5. Cross-validation and controlled experiments

Every candidate uses the same five stratified folds. This makes the comparisons fair.

In [10]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [11]:
logistic_scores = cross_val_score(
    model, X, y, cv=cv, scoring="accuracy"
)

print("Logistic regression fold accuracies:", logistic_scores)
print(f"Mean CV accuracy: {logistic_scores.mean():.4f}")
print(f"CV standard deviation: {logistic_scores.std():.4f}")

Logistic regression fold accuracies: [0.82681564 0.84269663 0.8258427  0.84269663 0.83707865]
Mean CV accuracy: 0.8350
CV standard deviation: 0.0074


In [12]:
tree_scores = cross_val_score(
    tree_model, X, y, cv=cv, scoring="accuracy"
)

random_forest_scores = cross_val_score(
    random_forest_model, X, y, cv=cv, scoring="accuracy"
)

print(f"Decision tree mean: {tree_scores.mean():.4f}")
print(f"Random forest mean: {random_forest_scores.mean():.4f}")

Decision tree mean: 0.8137
Random forest mean: 0.8159


In [13]:
features_without_title = [
    feature for feature in categorical_features
    if feature != "Title"
]

preprocessor_without_title = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, features_without_title)
])

model_without_title = Pipeline([
    ("preprocessor", preprocessor_without_title),
    ("classifier", LogisticRegression(max_iter=1000))
])

without_title_scores = cross_val_score(
    model_without_title, X, y, cv=cv, scoring="accuracy"
)

print(f"With Title mean: {logistic_scores.mean():.4f}")
print(f"Without Title mean: {without_title_scores.mean():.4f}")
print(f"Title contribution: {logistic_scores.mean() - without_title_scores.mean():+.4f}")

With Title mean: 0.8350
Without Title mean: 0.8171
Title contribution: +0.0180


In [14]:
features_without_fare_per_person = [
    feature for feature in numeric_features
    if feature != "FarePerPerson"
]

preprocessor_without_fare_per_person = ColumnTransformer([
    ("numeric", numeric_pipeline, features_without_fare_per_person),
    ("categorical", categorical_pipeline, categorical_features)
])

model_without_fare_per_person = Pipeline([
    ("preprocessor", preprocessor_without_fare_per_person),
    ("classifier", LogisticRegression(max_iter=1000))
])

without_fare_per_person_scores = cross_val_score(
    model_without_fare_per_person, X, y, cv=cv, scoring="accuracy"
)

print(f"With FarePerPerson mean: {logistic_scores.mean():.4f}")
print(f"Without FarePerPerson mean: {without_fare_per_person_scores.mean():.4f}")
print(
    f"FarePerPerson contribution: "
    f"{logistic_scores.mean() - without_fare_per_person_scores.mean():+.4f}"
)

With FarePerPerson mean: 0.8350
Without FarePerPerson mean: 0.8339
FarePerPerson contribution: +0.0011


## 6. Fixed validation-set error analysis

In [15]:
validation_model = clone(model)
validation_model.fit(X_train, y_train)
valid_predictions = validation_model.predict(X_valid)

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_valid, valid_predictions))

validation_errors = train.loc[
    X_valid.index,
    ["PassengerId", "Survived", "Sex", "Pclass", "Age", "Fare", "SibSp", "Parch", "Cabin"]
].copy()

validation_errors["Predicted"] = valid_predictions
validation_errors["SurvivalProbability"] = validation_model.predict_proba(X_valid)[:, 1]
validation_errors["ErrorType"] = np.select(
    [
        (validation_errors["Survived"] == 0) & (validation_errors["Predicted"] == 1),
        (validation_errors["Survived"] == 1) & (validation_errors["Predicted"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct"
)

validation_errors[validation_errors["ErrorType"] != "Correct"]

Confusion matrix [[TN, FP], [FN, TP]]:
[[100  10]
 [ 19  50]]


,PassengerId,Survived,Sex,Pclass,Age,Fare,SibSp,Parch,Cabin,Predicted,SurvivalProbability,ErrorType
553,554,1,male,3,22.0,7.2250,0,0,NaN,0,0.171579,False negative
559,560,1,female,3,36.0,17.4000,1,0,NaN,0,0.445678,False negative
536,537,0,male,1,45.0,26.5500,0,0,B38,1,0.585168,False positive
216,217,1,female,3,27.0,7.9250,0,0,NaN,0,0.478689,False negative
279,280,1,female,3,35.0,20.2500,1,1,NaN,0,0.385777,False negative
712,713,1,male,1,48.0,52.0000,1,0,C126,0,0.270495,False negative
455,456,1,male,3,29.0,7.8958,0,0,NaN,0,0.142756,False negative
297,298,0,female,1,2.0,151.5500,1,2,C22 C26,1,0.975231,False positive
502,503,0,female,3,NaN,7.6292,0,0,NaN,1,0.646670,False positive
17,18,1,male,2,NaN,13.0000,0,0,NaN,0,0.096741,False negative


In [16]:
rate_data = validation_errors.assign(
    actual_survivor=validation_errors["Survived"].eq(1),
    false_negative=(
        validation_errors["Survived"].eq(1)
        & validation_errors["Predicted"].eq(0)
    ),
    actual_death=validation_errors["Survived"].eq(0),
    false_positive=(
        validation_errors["Survived"].eq(0)
        & validation_errors["Predicted"].eq(1)
    ),
)

by_sex = rate_data.groupby("Sex").agg(
    actual_survivors=("actual_survivor", "sum"),
    false_negatives=("false_negative", "sum"),
    actual_deaths=("actual_death", "sum"),
    false_positives=("false_positive", "sum"),
)

by_sex["false_negative_rate_%"] = (
    100 * by_sex["false_negatives"] / by_sex["actual_survivors"]
).round(1)

by_sex["false_positive_rate_%"] = (
    100 * by_sex["false_positives"] / by_sex["actual_deaths"]
).round(1)

by_sex

,actual_survivors,false_negatives,actual_deaths,false_positives,false_negative_rate_%,false_positive_rate_%
Sex,,,,,,
female,45,3,16,7,6.7,43.8
male,24,16,94,3,66.7,3.2


In [17]:
male_survivors = validation_errors[
    (validation_errors["Sex"] == "male")
    & (validation_errors["Survived"] == 1)
].copy()

male_survivors.groupby("ErrorType")["SurvivalProbability"].describe()

,count,mean,std,min,25%,50%,75%,max
ErrorType,,,,,,,,
Correct,8.0,0.621039,0.112350,0.507801,0.549079,0.599935,0.649429,0.865625
False negative,16.0,0.213401,0.135465,0.035839,0.106419,0.172259,0.299719,0.450556


## 7. Final training and submission

The selected logistic-regression pipeline is now fitted on all labeled training rows. The submission is generated only here.

In [18]:
model.fit(X, y)

X_test = make_features(test)
final_predictions = model.predict(X_test)

final_submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_predictions
})

final_submission.to_csv("submission.csv", index=False)
final_submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## Conclusion

`Title` is the strongest feature-engineering improvement. `SexPclass` provides a smaller improvement. `SexAgeGroup` was tested separately and rejected because it reduced cross-validation accuracy. `FarePerPerson` is retained as a modest candidate, but its gain was small.

The remaining error pattern is that the model misses many male survivors. The analysis suggests these are often confidently misclassified, so changing the probability threshold alone is unlikely to solve the issue.